# ЛР 14. Передаваемый лабораторный конвейер

**Сущность:** Проверить работу с чужим CSV, сохранность строк и один вычислительный контракт для CLI и интерфейса.

**Данные:** molecules.csv; дополнительный ошибочный CSV без нужного столбца.

**Workflow:** CSV contract → ядро → журнал всех строк → паспорт → CLI → проверка передачи.

Перед запуском прочитайте [guideline](../guides/lab14.md) и [теорию](../../Module_theory/Module%2014_SDLS-workflow%20development_web%20app.md).

## 1. Прогноз

Запишите ожидаемое поведение и отрицательный контроль. Какое изменение входа должно изменить результат, а какое — сохранить?

In [1]:
from pathlib import Path
import sys, json, math, csv, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# WHY: relative discovery makes the notebook portable across machines.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Labs" / "chemistry.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите notebook внутри репозитория курса")
sys.path.insert(0, str(ROOT / "Labs"))
DATA = ROOT / "Labs" / "datasets"
OUT = ROOT / "Labs" / "results" / "lab14"
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()


## 2. Разбор и исполнение

1. Вызовите табличную функцию напрямую и проверьте все статусы.
2. Запустите тот же конвейер как отдельный CLI-процесс.
3. Сравните ID и hash отчётов, затем откройте локальный Streamlit при желании.

Ожидаемое: CLI и прямой вызов сохраняют 12 строк и hash; отсутствующий столбец/повторный ID отклоняются; Streamlit — тот же pipeline.

In [2]:
from pipeline import process_csv
import subprocess
data=(DATA/"molecules.csv").read_bytes()
rows,passport=process_csv(data)
assert len(rows)==12
assert sum(row["status"]=="invalid" for row in rows)==2
for bad in [b"id,wrong\nX,CCO\n",b"id,smiles\nX,CCO\nX,CCN\n"]:
    try: process_csv(bad)
    except ValueError: continue
    raise AssertionError("Неверный контракт принят")
process=subprocess.run([sys.executable,str(ROOT/"Labs/pipeline.py"),str(DATA/"molecules.csv"),"--out",str(OUT/"cli")],capture_output=True,text=True,check=True)
loaded=json.loads((OUT/"cli/report.json").read_text(encoding="utf-8"))
assert loaded["passport"]["input_sha256"]==passport["input_sha256"]
assert len(loaded["rows"])==12
display(pd.DataFrame(rows))
print("Интерфейс: python -m streamlit run Labs/app.py")
artifact=passport


[12:39:00] Initializing MetalDisconnector
[12:39:00] Running MetalDisconnector
[12:39:00] Initializing Normalizer
[12:39:00] Running Normalizer
[12:39:00] Initializing MetalDisconnector
[12:39:00] Running MetalDisconnector
[12:39:00] Initializing Normalizer
[12:39:00] Running Normalizer
[12:39:00] Running LargestFragmentChooser
[12:39:00] Running Uncharger
[12:39:00] Initializing MetalDisconnector
[12:39:00] Running MetalDisconnector
[12:39:00] Initializing Normalizer
[12:39:00] Running Normalizer
[12:39:00] Initializing MetalDisconnector
[12:39:00] Running MetalDisconnector
[12:39:00] Initializing Normalizer
[12:39:00] Running Normalizer
[12:39:00] Running LargestFragmentChooser
[12:39:00] Running Uncharger
[12:39:00] Initializing MetalDisconnector
[12:39:00] Running MetalDisconnector
[12:39:00] Initializing Normalizer
[12:39:00] Running Normalizer
[12:39:00] Initializing MetalDisconnector
[12:39:00] Running MetalDisconnector
[12:39:00] Initializing Normalizer
[12:39:00] Running Norma

,id,raw_smiles,parent_smiles,policy,MolWt,MolLogP,TPSA,HBD,HBA,ro5_violations,alerts,status,reason
0,M01,CCO,CCO,teaching-parent-v1;neutralize;tautomer=False,46.069,-0.0014,20.23,1.0,1.0,0.0,[],eligible,No listed alert; activity remains unknown
1,M02,OCC,CCO,teaching-parent-v1;neutralize;tautomer=False,46.069,-0.0014,20.23,1.0,1.0,0.0,[],eligible,No listed alert; activity remains unknown
2,M03,CC(=O)[O-].[Na+],CC(=O)O,teaching-parent-v1;neutralize;tautomer=False,60.052,0.0909,37.30,1.0,1.0,0.0,[],eligible,No listed alert; activity remains unknown
3,M04,CC(=O)O,CC(=O)O,teaching-parent-v1;neutralize;tautomer=False,60.052,0.0909,37.30,1.0,1.0,0.0,[],eligible,No listed alert; activity remains unknown
4,M05,Clc1ccccc1,Clc1ccccc1,teaching-parent-v1;neutralize;tautomer=False,112.559,2.3400,0.00,0.0,0.0,0.0,[],eligible,No listed alert; activity remains unknown
5,M06,Oc1ccccc1O,Oc1ccccc1O,teaching-parent-v1;neutralize;tautomer=False,110.112,1.0978,40.46,2.0,2.0,0.0,[catechol_A(92)],review,PAINS:catechol_A(92)
6,M07,c1ccccc1,c1ccccc1,teaching-parent-v1;neutralize;tautomer=False,78.114,1.6866,0.00,0.0,0.0,0.0,[],eligible,No listed alert; activity remains unknown
7,M08,not_a_smiles,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,invalid,Invalid SMILES
8,M09,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,invalid,SMILES must be a nonempty string
9,M10,C[C@H](O)C(=O)O,C[C@H](O)C(=O)O,teaching-parent-v1;neutralize;tautomer=False,90.078,-0.5482,57.53,2.0,2.0,0.0,[],eligible,No listed alert; activity remains unknown


Интерфейс: python -m streamlit run Labs/app.py


## 3. Проверяемая запись запуска

Benchmark фиксирует этот запуск. Время включает расчёт и создание артефактов, но не установку среды; это не сравнительный рейтинг оборудования.

In [3]:
elapsed = time.perf_counter()-started
run_record = {"lab": 14, "calculation_wall_seconds": elapsed, "result": artifact}
(OUT/"benchmark.json").write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding="utf-8")
print("Запуск сохранён:",OUT/"benchmark.json")


Запуск сохранён: /home/miakhlov/Computational-chemistry_course/Labs/results/lab14/benchmark.json


## 4. Самостоятельная тренировка

1. Передайте файл с повторным ID; объясните явный отказ.
2. Перенесите проект в новую папку и повторите запуск по README.
3. Как добавить другую политику, сохранив проверяемость результатов?

[Ответы после попытки](../solutions/lab14.md). Вернитесь к первому заданию через неделю.

## 5. Ваш научный вывод

1. Какой результат получен?
2. Какой вывод из него пока не следует?
3. Какой следующий эксперимент или контроль нужен?

Сохраните свой вариант notebook отдельно; исходный пример должен остаться воспроизводимым.